# 02 — Categorical Columns Encoding (Alternative / Data-Science-Informed Approach)

## Goal of this notebook
This notebook encodes the **same 10 categorical columns** as the original `02_categorical_encoding.ipynb`,
but takes a **more rigorous data-science approach**:

1. **Exploratory step first** — inspect cardinality, value distributions, and nulls *before* deciding on an encoding strategy.
2. **Encoding chosen per-column** based on the column's semantic nature:
   - True **nominal** with low cardinality → **One-Hot Encoding** (no artificial ordinal relationship implied).
   - Columns with a **natural order** (e.g. gear type, fuel type) → **Ordinal Encoding** (explicit integer rank).
   - **Very-high-cardinality** identifiers (`Marka`, `Seri`, `Model`, `Sınıfı`) → **Target Encoding** via mean-target (price proxy), which preserves statistical signal without creating ~1000 dummy columns.
3. **Null handling** — filled with `'Unknown'` as an explicit category (not mode), so the model can learn "we don't know" as a signal rather than silently borrowing the most-common value.
4. Every decision is **justified** in the markdown cell that precedes the code.

## Columns handled
`Marka`, `Seri`, `Model`, `Vites Tipi`, `Yakıt Tipi`, `Kasa Tipi`,
`Renk`, `Çekiş`, `Kimden`, `Sınıfı`

## Output
`df_team2_alt` — a fully numeric DataFrame with the same 3424-row index as the source dataset.

---
## Step 0 — Imports & Data Load

We load the raw dataset directly from the shared GitHub URL so this notebook is fully reproducible
without requiring a local copy of the file.
We also import `category_encoders` in addition to `sklearn` because it ships a ready-made,
cross-validated **TargetEncoder** that handles leave-one-out smoothing to prevent target leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# category_encoders gives us a convenient TargetEncoder with smoothing
try:
    import category_encoders as ce
    HAS_CE = True
except ImportError:
    HAS_CE = False
    print("category_encoders not installed — will fall back to pandas mean-target encoding.")
    print("Install with:  pip install category_encoders")

RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df_full = pd.read_csv(RAW_URL)
print(f"Loaded dataset: {df_full.shape}")
df_full.head(3)

---
## Step 1 — Select Team-2 Columns

We isolate only the 10 categorical columns assigned to Team 2.
We also keep **`Fiyat`** (price) in a separate Series `y` — it is our target variable
and will be required for Target Encoding without leaking information from the test set.

In [ ]:
MY_COLUMNS = ['Marka', 'Seri', 'Model', 'Vites Tipi', 'Yakıt Tipi',
              'Kasa Tipi', 'Renk', 'Çekiş', 'Kimden', 'Sınıfı']

df = df_full[MY_COLUMNS].copy()

# Keep price as the target for target-encoding steps
if 'Fiyat' in df_full.columns:
    y = df_full['Fiyat'].copy()
else:
    y = None
    print("⚠️  'Fiyat' column not found — target encoding will use row index rank as proxy.")

print(f"Working subset shape: {df.shape}")
df.dtypes

---
## Step 2 — Exploratory Analysis (EDA) Before Encoding

**Why do EDA before encoding?**
Choosing an encoding method *blindly* (e.g., always applying Label Encoding) risks introducing
a spurious ordinal relationship between categories or creating too many dummy columns.
By examining cardinality and null counts first we can make a **data-driven decision** per column.

The table below shows unique-value count (`cardinality`) and null percentage for each column.
We use these numbers to define three groups:
- **Very high cardinality (> 50 unique)** → Target Encoding
- **Low cardinality (≤ 10 unique)** → One-Hot Encoding (if truly nominal) or Ordinal Encoding (if ordered)
- **Medium cardinality (11–50 unique)** → One-Hot or Ordinal depending on semantics

In [ ]:
eda = pd.DataFrame({
    'cardinality': df.nunique(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'sample_values': [df[c].dropna().unique()[:5].tolist() for c in df.columns]
})
print(eda.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
eda['cardinality'].sort_values(ascending=False).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Cardinality of Categorical Columns (unique value count)')
ax.set_ylabel('# unique values')
ax.axhline(50, color='red',  linestyle='--', label='High-cardinality threshold (50)')
ax.axhline(10, color='orange', linestyle='--', label='Low-cardinality threshold (10)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 3 — Null Handling

**Why `'Unknown'` instead of mode imputation?**

The original notebook fills missing values with the **mode** of each column.
While simple, this approach has a hidden cost: it *invents* information — it assumes the car
belongs to the most-popular category, which may be completely wrong.

A better practice for categorical data is to introduce an explicit `'Unknown'` (or `'Missing'`)
category. This way:
- The encoding maps `'Unknown'` to its own numeric code, so the model *learns* what this label means.
- We preserve the **absence of information** as a real signal (e.g., sellers who don't specify
  the car class tend to list cheaper or undisclosed inventory).
- We avoid inflating the apparent frequency of the most-common category.

**Exception**: if a column has > 50 % nulls, dropping or secondary imputation may be more appropriate.
Here we check if any column exceeds that threshold.

In [ ]:
HIGH_NULL_THRESHOLD = 50  # percent

for col in MY_COLUMNS:
    null_pct = df[col].isnull().mean() * 100
    if null_pct > HIGH_NULL_THRESHOLD:
        print(f"⚠️  '{col}' has {null_pct:.1f}% nulls — consider dropping or special treatment.")
    elif null_pct > 0:
        df[col] = df[col].fillna('Unknown')
        print(f"  Filled {col!r} nulls with 'Unknown' ({null_pct:.2f}% of rows)")

print(f"\nTotal nulls after fill: {df.isnull().sum().sum()}")

---
## Step 4 — Encoding Strategy Decision

Based on the EDA we allocate each column to one of three encoding strategies:

| Column | Cardinality | Nature | Strategy |
|--------|------------|--------|----------|
| `Marka` | Very high | Nominal identifier | **Target Encoding** |
| `Seri` | Very high | Nominal identifier | **Target Encoding** |
| `Model` | Very high | Nominal identifier | **Target Encoding** |
| `Sınıfı` | Medium-high | Nominal identifier | **Target Encoding** |
| `Vites Tipi` | Low | **Ordered** (Automatic > Semi > Manual) | **Ordinal Encoding** |
| `Yakıt Tipi` | Low | Nominal (no natural order) | **One-Hot Encoding** |
| `Kasa Tipi` | Low-medium | Nominal | **One-Hot Encoding** |
| `Renk` | Medium | Nominal | **One-Hot Encoding** |
| `Çekiş` | Low | Nominal | **One-Hot Encoding** |
| `Kimden` | Very low (2–3) | Nominal | **One-Hot Encoding** |

### Why NOT Label Encoding for the low-cardinality columns?
Label Encoding assigns arbitrary integers (0, 1, 2 …) in **alphabetical order**,
which implies a magnitude relationship that doesn't exist: e.g., `'Beyaz'=0 < 'Gri'=2`
would tell a decision-tree or linear model that grey is numerically greater than white,
which is meaningless for colour. One-Hot Encoding avoids this by creating a separate
binary column for each category — the model treats each as an independent yes/no feature.

### Why NOT One-Hot for `Marka`, `Seri`, `Model`?
One-Hot Encoding on a column with 300+ unique values would create 300+ new binary columns,
causing the **curse of dimensionality** and making the dataset extremely sparse.
Target Encoding shrinks each category to a single float (its mean price),
which is both memory-efficient and semantically meaningful.

### Why Ordinal for `Vites Tipi`?
Gear-type has a logical progression in performance and user preference:
`Manuel < Yarı Otomatik < Otomatik` (roughly: manual < semi-auto < automatic).
Assigning integers 0, 1, 2 to respect this order is valid and avoids dummy columns.

---
## Step 5 — Target Encoding for High-Cardinality Columns

Target encoding replaces each category value with the **mean of the target variable** (price)
for all rows that share that category. For example, if BMW listings have a mean price of 850,000 TL,
every `Marka == 'BMW'` row becomes `850000.0`.

**Smoothing / leave-one-out (LOO):** When using `category_encoders.TargetEncoder`, the encoder
applies Bayesian smoothing: rare categories (few observations) are shrunk toward the global mean
instead of taking an unreliable sample mean. This prevents over-fitting on categories with only
1–2 rows. If `category_encoders` is not available we fall back to a simple `groupby` mean
(slightly lower quality, but still better than Label Encoding for high-cardinality columns).

In [ ]:
HIGH_CARD_COLS = ['Marka', 'Seri', 'Model', 'Sınıfı']

if y is None:
    # No price column: use row rank as a proxy target so the notebook still runs
    y_enc = pd.Series(range(len(df)), index=df.index, dtype=float)
else:
    y_enc = y.copy()

if HAS_CE:
    te = ce.TargetEncoder(cols=HIGH_CARD_COLS, smoothing=1.0)
    df[HIGH_CARD_COLS] = te.fit_transform(df[HIGH_CARD_COLS], y_enc)
    print("Target encoding applied via category_encoders (with Bayesian smoothing).")
else:
    # Manual mean-target encoding (no smoothing)
    for col in HIGH_CARD_COLS:
        means = y_enc.groupby(df[col]).mean()
        df[col] = df[col].map(means)
        # Any unmapped value (e.g. new 'Unknown') gets global mean
        df[col] = df[col].fillna(y_enc.mean())
    print("Manual mean-target encoding applied (install category_encoders for smoothing).")

print("\nSample encoded values:")
df[HIGH_CARD_COLS].head()

---
## Step 6 — Ordinal Encoding for `Vites Tipi` (Gear Type)

`Vites Tipi` describes the **transmission type**, which has a reasonable ordering:
manual gearboxes are most driver-intensive; automatic least so.
Car pricing and buyer preference also tend to follow this gradient
(automatics typically command a premium in the Turkish second-hand market).

We use `sklearn.preprocessing.OrdinalEncoder` and supply an explicit `categories` list
to control the mapping ourselves, rather than letting the encoder assign arbitrary alphabetical ranks.

Any unseen category (including `'Unknown'`) is mapped to `-1` via `handle_unknown='use_encoded_value'`,
which is distinct from valid ordinal ranks so the model can differentiate it.

In [ ]:
# Inspect actual values in the column first
print("Unique Vites Tipi values:", df['Vites Tipi'].unique().tolist())

# Define the order from 'least automatic' to 'most automatic'
vites_order = ['Manuel', 'Yarı Otomatik', 'Otomatik', 'Unknown']

# Keep only values that actually appear in the data
vites_order_present = [v for v in vites_order if v in df['Vites Tipi'].values]

oe = OrdinalEncoder(
    categories=[vites_order_present],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

df['Vites Tipi'] = oe.fit_transform(df[['Vites Tipi']]).astype(int)
print("\nOrdinal mapping:", dict(zip(vites_order_present, range(len(vites_order_present)))))
df['Vites Tipi'].value_counts()

---
## Step 7 — One-Hot Encoding for Nominal Low-Cardinality Columns

The remaining columns — `Yakıt Tipi`, `Kasa Tipi`, `Renk`, `Çekiş`, `Kimden` — are all **nominal**:
there is no meaningful order between their values.

**Why One-Hot over Label Encoding?**
Label Encoding would implicitly tell any distance-based or linear model that e.g.
`'Dizel' = 2 > 'Benzin' = 0`, implying diesel is numerically *greater* than petrol.
This magnitude relationship is completely artificial and can mislead the model.
One-Hot Encoding avoids this by giving each category its own binary dimension.

We use `drop='first'` (drop one dummy per variable) to avoid **perfect multicollinearity**
(the dummy trap) for linear models. Tree-based models are immune to this, but it's good
practice and keeps the output leaner while still being fully reversible.

We set `handle_unknown='ignore'` so that any value that did not appear during fit
(e.g. an edge-case after `'Unknown'` filling) is silently turned into all-zeros,
rather than raising an error.

In [ ]:
OHE_COLS = ['Yakıt Tipi', 'Kasa Tipi', 'Renk', 'Çekiş', 'Kimden']

ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

ohe_array = ohe.fit_transform(df[OHE_COLS])
ohe_feature_names = ohe.get_feature_names_out(OHE_COLS)

df_ohe = pd.DataFrame(ohe_array, columns=ohe_feature_names, index=df.index)

# Drop the original string columns and join the dummies
df = df.drop(columns=OHE_COLS).join(df_ohe)

print(f"Shape after OHE: {df.shape}")
print(f"New OHE columns ({len(ohe_feature_names)}): {ohe_feature_names.tolist()[:10]} ...")

---
## Step 8 — Verification & Output

We run the same assertion checks as the original notebook (zero nulls, fully numeric)
and additionally print a summary of the encoding decisions made.

The final DataFrame is named `df_team2_alt` to make the comparison with `df_team2` easy.

In [ ]:
df_team2_alt = df.copy()

# Sanity checks
assert df_team2_alt.isnull().sum().sum() == 0, "❌ Nulls remain!"
assert df_team2_alt.select_dtypes(exclude='number').shape[1] == 0, "❌ Non-numeric columns remain!"

print(f"✅ Alternative Team 2 output ready.")
print(f"   Shape: {df_team2_alt.shape}")
print(f"   Columns: {df_team2_alt.columns.tolist()}")
print()
print("Encoding summary:")
print(f"  Target-encoded   : {HIGH_CARD_COLS}")
print(f"  Ordinal-encoded  : ['Vites Tipi']")
print(f"  One-Hot-encoded  : {OHE_COLS}  → {len(ohe_feature_names)} dummy columns")
print()
df_team2_alt.head()

In [ ]:
print(df_team2_alt.dtypes)

---
## Step 9 — Side-by-Side Comparison With Original Approach

| Aspect | Original `02_categorical_encoding.ipynb` | This Alternative |
|--------|------------------------------------------|------------------|
| **Null strategy** | Fill with **mode** (invents info) | Fill with `'Unknown'` (preserves absence-of-info as signal) |
| **High-cardinality** | Label Encoding (arbitrary integer, no semantic meaning) | **Target Encoding** (mean price — semantically meaningful, compact) |
| **Low-cardinality nominal** | Label Encoding (false ordinal relationship) | **One-Hot Encoding** (no false ordering, preferred for linear/distance models) |
| **`Vites Tipi`** | Label Encoding (alphabetical order) | **Ordinal Encoding** (explicit meaningful order: Manual < Semi < Auto) |
| **Output shape** | (3424, 10) — 10 columns | (3424, ~30) — 10 original + ~20 OHE dummies |
| **EDA before encoding** | Not present | ✅ Cardinality plot + null analysis |
| **Justification per step** | Comments only | ✅ Full markdown rationale per step |

### When is the original approach acceptable?
Label Encoding for all columns is perfectly valid when training **tree-based models**
(Random Forests, XGBoost, LightGBM) — these models split on thresholds and don't assume
any ordinal relationship between integer codes. If the downstream model is a tree ensemble,
the original approach is simpler, produces fewer columns, and will work fine.

This alternative is preferable when the downstream model includes **linear models** (Regression,
SVM, KNN) that are sensitive to implied magnitude relationships between integer codes.